In [2]:
!pip install biopython

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 31.8 MB/s eta 0:00:00


In [1]:
import Bio
print(Bio.__version__)

1.88


In [2]:
test_pmids = [
    "41467193",
    "41465382",
    "41400063",
    "41472084",
    "41224192"
]

test_pmids

['41467193', '41465382', '41400063', '41472084', '41224192']

In [4]:
from Bio import Entrez

Entrez.email = "samitkumardas321@gmail.com"

handle = Entrez.efetch(
    db="pubmed",
    id=test_pmids,
    retmode="xml"
)

records = Entrez.read(handle)

len(records["PubmedArticle"])

5

In [5]:
for article in records["PubmedArticle"]:
    
    medline = article["MedlineCitation"]
    
    pmid = str(medline["PMID"])
    title = str(medline["Article"]["ArticleTitle"])
    
    abstract_parts = medline["Article"].get("Abstract", {}).get("AbstractText", [])
    
    abstract = " ".join(str(part) for part in abstract_parts)
    
    print("=" * 100)
    print("PMID:", pmid)
    print("TITLE:", title)
    print("ABSTRACT LENGTH:", len(abstract))
    print("ABSTRACT:", abstract[:500])

PMID: 41467193
TITLE: <i>In vivo</i> and <i>in vitro</i> experimental models for urolithiasis pathophysiology research.
ABSTRACT LENGTH: 2403
ABSTRACT: This review aims to investigate and establish potential <i>in vitro</i> and <i>in vivo</i> models for evaluating the anti-urolithiatic activity of therapeutic agents, exploring experimental approaches that can reliably simulate human stone formation and effectively assess the efficacy of anti-urolithiatic compounds. Multiple <i>in vitro</i> and <i>in vivo</i> approaches were explored. <i>In vitro</i> methods included the​​ estimation of calcium oxalate by titrimetry, nucleation assays, aggregati
PMID: 41465382
TITLE: <i>MTAP</i> Deletion as a Therapeutic Vulnerability in Cancer: From Molecular Mechanism to Clinical Targeting.
ABSTRACT LENGTH: 1428
ABSTRACT: The <i>MTAP</i> (methylthioadenosine phosphorylase) gene, located on chromosome 9p21, plays a crucial role in the methionine salvage pathway and is frequently co-deleted with <i>CDKN

In [6]:
for article in records["PubmedArticle"]:
    
    medline = article["MedlineCitation"]
    pmid = str(medline["PMID"])
    
    if pmid == "41467193":
        
        abstract_parts = medline["Article"].get("Abstract", {}).get("AbstractText", [])
        
        abstract = " ".join(str(part) for part in abstract_parts)
        
        print("PMID:", pmid)
        print("ABSTRACT LENGTH:", len(abstract))
        print("\nFULL ABSTRACT:\n")
        print(abstract)

PMID: 41467193
ABSTRACT LENGTH: 2403

FULL ABSTRACT:

This review aims to investigate and establish potential <i>in vitro</i> and <i>in vivo</i> models for evaluating the anti-urolithiatic activity of therapeutic agents, exploring experimental approaches that can reliably simulate human stone formation and effectively assess the efficacy of anti-urolithiatic compounds. Multiple <i>in vitro</i> and <i>in vivo</i> approaches were explored. <i>In vitro</i> methods included the​​ estimation of calcium oxalate by titrimetry, nucleation assays, aggregation assays, turbidimetric assays, and electron microscopy studies. Artificial stone models such as BegoStone and Ultracal 30 were fabricated to mimic the physicochemical characteristics of urinary calculi. <i>In vivo</i> models included ethylene glycol-induced, calcium oxalate/ammonium oxalate-induced, diet-induced, and infection-related models in rodents. Additionally, genetically modified animal models such as <i>TRPV5</i> knockout, <i>CLDN1

In [8]:
import pandas as pd
df = pd.read_csv("pubmed_medical_affairs_3layer_corpus.csv")

In [9]:
original_abstract = df.loc[
    df["pmid"] == 41467193, "abstract"
].iloc[0]

print("Original CSV length:", len(original_abstract))
print("PubMed abstract length:", len(abstract))

Original CSV length: 81
PubMed abstract length: 2403


In [10]:
print("ORIGINAL CSV ABSTRACT:")
print(original_abstract)

print("\n" + "="*100 + "\n")

print("PUBMED ABSTRACT:")
print(abstract)

ORIGINAL CSV ABSTRACT:
This review aims to investigate and establish potential Multiple A combination of


PUBMED ABSTRACT:
This review aims to investigate and establish potential <i>in vitro</i> and <i>in vivo</i> models for evaluating the anti-urolithiatic activity of therapeutic agents, exploring experimental approaches that can reliably simulate human stone formation and effectively assess the efficacy of anti-urolithiatic compounds. Multiple <i>in vitro</i> and <i>in vivo</i> approaches were explored. <i>In vitro</i> methods included the​​ estimation of calcium oxalate by titrimetry, nucleation assays, aggregation assays, turbidimetric assays, and electron microscopy studies. Artificial stone models such as BegoStone and Ultracal 30 were fabricated to mimic the physicochemical characteristics of urinary calculi. <i>In vivo</i> models included ethylene glycol-induced, calcium oxalate/ammonium oxalate-induced, diet-induced, and infection-related models in rodents. Additionally, gene

In [11]:
import pandas as pd
import numpy as np
import requests
import xml.etree.ElementTree as ET
import re
import time
from html import unescape

# Load original dataset
df = pd.read_csv("pubmed_medical_affairs_3layer_corpus.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

Dataset shape: (21256, 6)
Columns: ['pmid', 'title', 'abstract', 'year', 'journal', 'layer']


In [12]:
def clean_pubmed_text(element):
    """
    Extract all textual content from a PubMed XML element,
    removing XML/HTML formatting tags.
    """
    
    if element is None:
        return ""
    
    text = "".join(element.itertext())
    
    # Decode HTML entities
    text = unescape(text)
    
    # Remove excessive whitespace
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()

In [15]:
def extract_abstract(article):
    """
    Extract complete abstract from a PubMedArticle XML record.
    Preserves structured abstract section labels where available.
    """
    
    article_elem = article.find(".//MedlineCitation/Article")
    
    if article_elem is None:
        return ""
    
    abstract_elem = article_elem.find("Abstract")
    
    if abstract_elem is None:
        return ""
    
    sections = []
    
    for abstract_text in abstract_elem.findall("AbstractText"):
        
        text = clean_pubmed_text(abstract_text)
        
        if not text:
            continue
        
        label = abstract_text.attrib.get("Label", "").strip()
        
        if label:
            sections.append(f"{label}: {text}")
        else:
            sections.append(text)
    
    return " ".join(sections).strip()

In [17]:
test_pmid = "41467193"

url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

params = {
    "db": "pubmed",
    "id": test_pmid,
    "retmode": "xml",
    "rettype": "abstract",
    "tool": "masters_thesis_nlp",
    "email": "samitkumardas321@gmailcom"
}

response = requests.get(url, params=params)

print("Status code:", response.status_code)

root = ET.fromstring(response.content)

article = root.find(".//PubmedArticle")

repaired_abstract = extract_abstract(article)

print("PMID:", test_pmid)
print("Repaired abstract length:", len(repaired_abstract))
print("\nRepaired abstract:\n")
print(repaired_abstract)

Status code: 200
PMID: 41467193
Repaired abstract length: 2346

Repaired abstract:

OBJECTIVE: This review aims to investigate and establish potential in vitro and in vivo models for evaluating the anti-urolithiatic activity of therapeutic agents, exploring experimental approaches that can reliably simulate human stone formation and effectively assess the efficacy of anti-urolithiatic compounds. METHODS: Multiple in vitro and in vivo approaches were explored. In vitro methods included the​​ estimation of calcium oxalate by titrimetry, nucleation assays, aggregation assays, turbidimetric assays, and electron microscopy studies. Artificial stone models such as BegoStone and Ultracal 30 were fabricated to mimic the physicochemical characteristics of urinary calculi. In vivo models included ethylene glycol-induced, calcium oxalate/ammonium oxalate-induced, diet-induced, and infection-related models in rodents. Additionally, genetically modified animal models such as TRPV5 knockout, CLDN14 

In [18]:
original_abstract = df.loc[
    df["pmid"] == 41467193, "abstract"
].iloc[0]

print("Original CSV length :", len(str(original_abstract)))
print("Repaired length     :", len(repaired_abstract))

print("\n" + "="*80)

print("ORIGINAL:")
print(original_abstract)

print("\n" + "="*80)

print("REPAIRED:")
print(repaired_abstract)

Original CSV length : 81
Repaired length     : 2346

ORIGINAL:
This review aims to investigate and establish potential Multiple A combination of

REPAIRED:
OBJECTIVE: This review aims to investigate and establish potential in vitro and in vivo models for evaluating the anti-urolithiatic activity of therapeutic agents, exploring experimental approaches that can reliably simulate human stone formation and effectively assess the efficacy of anti-urolithiatic compounds. METHODS: Multiple in vitro and in vivo approaches were explored. In vitro methods included the​​ estimation of calcium oxalate by titrimetry, nucleation assays, aggregation assays, turbidimetric assays, and electron microscopy studies. Artificial stone models such as BegoStone and Ultracal 30 were fabricated to mimic the physicochemical characteristics of urinary calculi. In vivo models included ethylene glycol-induced, calcium oxalate/ammonium oxalate-induced, diet-induced, and infection-related models in rodents. Addition

In [20]:
BATCH_SIZE = 200

# Make sure PMIDs are strings
pmids = df["pmid"].astype(str).tolist()

all_abstracts = {}
failed_batches = []

total_batches = (len(pmids) + BATCH_SIZE - 1) // BATCH_SIZE

print("Total PMIDs:", len(pmids))
print("Total batches:", total_batches)

url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

for batch_number, start in enumerate(range(0, len(pmids), BATCH_SIZE), start=1):
    
    batch_pmids = pmids[start:start + BATCH_SIZE]
    
    params = {
        "db": "pubmed",
        "id": ",".join(batch_pmids),
        "retmode": "xml",
        "rettype": "abstract",
        "tool": "masters_thesis_nlp",
        "email": "samitkumardas321@gmail.com"
    }
    
    success = False
    
    for attempt in range(3):
        
        try:
            response = requests.get(
                url,
                params=params,
                timeout=60
            )
            
            if response.status_code == 200:
                success = True
                break
            
            else:
                print(
                    f"Batch {batch_number}: "
                    f"HTTP {response.status_code}. "
                    f"Retrying..."
                )
                
                time.sleep(2 ** attempt)
        
        except Exception as e:
            
            print(
                f"Batch {batch_number}: "
                f"Error: {e}. Retrying..."
            )
            
            time.sleep(2 ** attempt)
    
    if not success:
        
        failed_batches.append(batch_number)
        print(f"FAILED BATCH: {batch_number}")
        continue
    
    try:
        
        root = ET.fromstring(response.content)
        
        articles = root.findall(".//PubmedArticle")
        
        for article in articles:
            
            pmid_element = article.find(".//MedlineCitation/PMID")
            
            if pmid_element is None:
                continue
            
            pmid = pmid_element.text
            
            abstract = extract_abstract(article)
            
            all_abstracts[str(pmid)] = abstract
    
    except Exception as e:
        
        failed_batches.append(batch_number)
        
        print(
            f"XML parsing failed for batch {batch_number}: {e}"
        )
    
    # Respect NCBI request rate
    time.sleep(0.4)
    
    if batch_number % 10 == 0 or batch_number == total_batches:
        
        print(
            f"Completed {batch_number}/{total_batches} batches "
            f"({len(all_abstracts)} abstracts retrieved)"
        )

print("\nRetrieval completed.")
print("Abstracts retrieved:", len(all_abstracts))
print("Failed batches:", failed_batches)

Total PMIDs: 21256
Total batches: 107
Completed 10/107 batches (1996 abstracts retrieved)
Completed 20/107 batches (3996 abstracts retrieved)


KeyboardInterrupt: 

In [21]:
BATCH_SIZE = 200

pmids = df["pmid"].astype(str).tolist()

all_abstracts = {}
failed_batches = []

total_batches = (len(pmids) + BATCH_SIZE - 1) // BATCH_SIZE

print("Total PMIDs:", len(pmids))
print("Total batches:", total_batches)
print("Starting PubMed retrieval...\n")

url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

for batch_number, start in enumerate(
    range(0, len(pmids), BATCH_SIZE), start=1
):

    batch_pmids = pmids[start:start + BATCH_SIZE]

    params = {
        "db": "pubmed",
        "id": ",".join(batch_pmids),
        "retmode": "xml",
        "rettype": "abstract",
        "tool": "masters_thesis_nlp",
        "email": "YOUR_EMAIL@example.com"
    }

    success = False

    for attempt in range(3):

        try:

            response = requests.get(
                url,
                params=params,
                timeout=60
            )

            if response.status_code == 200:
                success = True
                break

            print(
                f"Batch {batch_number}: HTTP "
                f"{response.status_code} - retrying..."
            )

            time.sleep(2 ** attempt)

        except Exception as e:

            print(
                f"Batch {batch_number}: {e} - retrying..."
            )

            time.sleep(2 ** attempt)

    if not success:

        failed_batches.append(batch_number)

        print(
            f"❌ Batch {batch_number} FAILED"
        )

        continue

    try:

        root = ET.fromstring(response.content)

        articles = root.findall(".//PubmedArticle")

        batch_count = 0

        for article in articles:

            pmid_element = article.find(
                ".//MedlineCitation/PMID"
            )

            if pmid_element is None:
                continue

            pmid = str(pmid_element.text)

            abstract = extract_abstract(article)

            all_abstracts[pmid] = abstract

            batch_count += 1

        print(
            f"Batch {batch_number}/{total_batches} ✓ "
            f"| Records: {batch_count} "
            f"| Total retrieved: {len(all_abstracts)}"
        )

    except Exception as e:

        failed_batches.append(batch_number)

        print(
            f"❌ XML parsing failed for batch "
            f"{batch_number}: {e}"
        )

    time.sleep(0.4)


print("\n==============================")
print("RETRIEVAL COMPLETED")
print("==============================")
print("Original PMIDs:", len(pmids))
print("Abstracts retrieved:", len(all_abstracts))
print("Failed batches:", failed_batches)

Total PMIDs: 21256
Total batches: 107
Starting PubMed retrieval...

Batch 1/107 ✓ | Records: 200 | Total retrieved: 200
Batch 2/107 ✓ | Records: 200 | Total retrieved: 400
Batch 3/107 ✓ | Records: 200 | Total retrieved: 600
Batch 4/107 ✓ | Records: 200 | Total retrieved: 800
Batch 5/107 ✓ | Records: 197 | Total retrieved: 997
Batch 6/107 ✓ | Records: 200 | Total retrieved: 1197
Batch 7/107 ✓ | Records: 200 | Total retrieved: 1397
Batch 8/107 ✓ | Records: 199 | Total retrieved: 1596
Batch 9/107 ✓ | Records: 200 | Total retrieved: 1796
Batch 10/107 ✓ | Records: 200 | Total retrieved: 1996
Batch 11/107 ✓ | Records: 200 | Total retrieved: 2196
Batch 12/107 ✓ | Records: 200 | Total retrieved: 2396
Batch 13/107 ✓ | Records: 200 | Total retrieved: 2596
Batch 14/107 ✓ | Records: 200 | Total retrieved: 2796
Batch 15/107 ✓ | Records: 200 | Total retrieved: 2996
Batch 16/107 ✓ | Records: 200 | Total retrieved: 3196
Batch 17/107 ✓ | Records: 200 | Total retrieved: 3396
Batch 18/107 ✓ | Records: 20

In [31]:
# Create a backup of the current abstract column
df["abstract_original"] = df["abstract"]

# Make sure PMIDs are strings
df["pmid"] = df["pmid"].astype(str)

# Replace the existing abstract with the PubMed-retrieved version
# wherever a complete PubMed abstract was successfully retrieved

df["abstract_retrieved"] = df["pmid"].map(all_abstracts)

# Count how many PubMed abstracts were successfully matched
retrieved_count = df["abstract_retrieved"].notna().sum()

print("Total records:", len(df))
print("PubMed abstracts matched:", retrieved_count)
print("PubMed abstracts not matched:", len(df) - retrieved_count)

# Replace only where a PubMed abstract exists and is non-empty
valid_retrieved = (
    df["abstract_retrieved"].notna() &
    (df["abstract_retrieved"].astype(str).str.strip() != "")
)

df.loc[valid_retrieved, "abstract"] = df.loc[valid_retrieved, "abstract_retrieved"]

print("Abstracts replaced with PubMed versions:",
      valid_retrieved.sum())

Total records: 21256
PubMed abstracts matched: 21241
PubMed abstracts not matched: 15
Abstracts replaced with PubMed versions: 21239


In [33]:
# Verify abstract completeness after PubMed correction

df["abstract_length"] = df["abstract"].fillna("").astype(str).str.len()

print("Total records:", len(df))
print("Empty abstracts:", (df["abstract_length"] == 0).sum())
print("Abstracts < 200 characters:", (df["abstract_length"] < 200).sum())

print("\nShortest abstracts:")
print(
    df.loc[
        df["abstract_length"].nsmallest(10).index,
        ["pmid", "title", "abstract_length", "layer"]
    ]
)

Total records: 21256
Empty abstracts: 0
Abstracts < 200 characters: 1

Shortest abstracts:
           pmid                                              title  \
16216  41436046  [The applications and challenges of rapid dete...   
15647  41468091  Adjuvant Chemoradiotherapy Versus Radiotherapy...   
9004   41101910              Orthobiologics for Meniscus Injuries.   
13341  41079276  Endoscopic strategies for the management of lo...   
6943   41180001  Efficacy and Safety of Bictegravir Plus Lenaca...   
1811   41392927  R WE ready for reimbursement? A round-up of de...   
8778   41109726  Advantages and Disadvantages of the Arthroscop...   
15182  40991743   Attaching the Sacrocolpopexy Mesh to the Sacrum.   
1886   41389712  TL1A blockade in inflammatory bowel diseases: ...   
5566   41233099  Updates on Multimodal Analgesia for Orthopedic...   

       abstract_length                     layer  
16216              127   L3_comparative_critical  
15647              313   L3_comparat

In [34]:
# Investigate the unusually short abstract

short_record = df[df["abstract_length"] < 200]

for _, row in short_record.iterrows():
    print("=" * 100)
    print("PMID:", row["pmid"])
    print("TITLE:", row["title"])
    print("YEAR:", row["year"])
    print("JOURNAL:", row["journal"])
    print("LAYER:", row["layer"])
    print("ABSTRACT LENGTH:", row["abstract_length"])
    print("\nABSTRACT:")
    print(row["abstract"])

PMID: 41436046
TITLE: [The applications and challenges of rapid detection technologies in forensic toxicology].
YEAR: 2025.0
JOURNAL: Fa yi xue za zhi
LAYER: L3_comparative_critical
ABSTRACT LENGTH: 127

ABSTRACT:
法医毒物学作为法医学的重要分支，是连接中毒事件调查、司法公正裁决与公共安全保障的关键技术支撑，其核心使命在于精准查明中毒案件中的毒物种类、代谢路径、暴露剂量及中毒机制，最终为案件侦破提供方向指引、为司法审判提供科学证据、为突发公共安全事件处置提供决策依据


In [32]:
# Step 7: Check the retrieved abstract dictionary

print("Number of abstracts retrieved:", len(all_abstracts))

# Check the first 5 retrieved records
for i, (pmid, abstract) in enumerate(all_abstracts.items()):
    print("\nPMID:", pmid)
    print("Abstract length:", len(abstract))
    print("Abstract preview:", abstract[:300])

    if i == 4:
        break

Number of abstracts retrieved: 21241

PMID: 41483736
Abstract length: 1796
Abstract preview: Antibody-drug conjugates (ADCs) represent a rapidly evolving class of therapeutics in non-small cell lung cancer (NSCLC), offering targeted delivery of cytotoxic payloads to tumor cells while minimizing off-target toxicity. Although ADCs have demonstrated promising results in NSCLC, their efficacy i

PMID: 41483728
Abstract length: 1528
Abstract preview: INTRODUCTION: Delirium is a neuropsychiatric condition, characterized by cloudy state of consciousness. Though commonly treated by haloperidol, quetiapine offers its own distinct benefits. A focused systematic review and meta-analysis was performed comparing efficacy of quetiapine with haloperidol i

PMID: 41483727
Abstract length: 2300
Abstract preview: BACKGROUND: Radiology faces an unprecedented workload crisis, creating demand for AI solutions to enhance efficiency and quality. Vision-language models (VLMs) represent a paradigm shift from n

In [23]:
# Step 8: Identify PMIDs for which PubMed did not return an abstract

retrieved_pmids = set(all_abstracts.keys())
original_pmids = set(df["pmid"].astype(str))

missing_pmids = sorted(original_pmids - retrieved_pmids)

print("Original PMIDs:", len(original_pmids))
print("Retrieved PMIDs:", len(retrieved_pmids))
print("Missing PMIDs:", len(missing_pmids))

print("\nPMIDs without retrieved abstracts:")
print(missing_pmids)

Original PMIDs: 21256
Retrieved PMIDs: 21241
Missing PMIDs: 15

PMIDs without retrieved abstracts:
['41219071', '41412986', '41412990', '41433266', '41433272', '41433318', '41433361', '41433371', '41433458', '41433513', '41433740', '41433764', '41434179', '41468071', '41468091']


In [25]:
# Step 9: Check the 15 PMIDs that were not retrieved

import requests
import time

missing_ids = ",".join(missing_pmids)

url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"

params = {
    "db": "pubmed",
    "id": missing_ids,
    "retmode": "xml",
    "rettype": "abstract"
}

response = requests.get(url, params=params, timeout=60)

print("HTTP status:", response.status_code)

if response.status_code == 200:
    records_missing = response.text
    print("PubMed response received successfully.")
    print("Response length:", len(records_missing))
else:
    print("Request failed.")

HTTP status: 200
PubMed response received successfully.
Response length: 206


In [26]:
print(records_missing)

<?xml version="1.0" ?>
<!DOCTYPE PubmedArticleSet PUBLIC "-//NLM//DTD PubMedArticle, 1st January 2025//EN" "https://dtd.nlm.nih.gov/ncbi/pubmed/out/pubmed_250101.dtd">
<PubmedArticleSet>
</PubmedArticleSet>


In [27]:
# Step 11: Replace incomplete abstracts with the newly retrieved PubMed abstracts

# Make a copy of the original dataset
df_cleaned = df.copy()

# Make sure PMID is treated consistently as a string
df_cleaned["pmid"] = df_cleaned["pmid"].astype(str)

# Replace the abstract with the newly retrieved PubMed abstract
df_cleaned["abstract"] = df_cleaned["pmid"].map(all_abstracts)

# For the 15 PMIDs that were not retrieved,
# preserve the original abstract rather than replacing it with NaN
original_abstracts = df["abstract"].copy()

df_cleaned["abstract"] = df_cleaned["abstract"].fillna(original_abstracts)

# Check the resulting dataset
print("Original dataset shape:", df.shape)
print("Cleaned dataset shape:", df_cleaned.shape)

print("\nMissing values in cleaned dataset:")
print(df_cleaned.isnull().sum())

Original dataset shape: (21256, 6)
Cleaned dataset shape: (21256, 6)

Missing values in cleaned dataset:
pmid         0
title       42
abstract     0
year         8
journal      0
layer        0
dtype: int64


In [28]:
# Step 12: Identify potentially incomplete or suspicious abstracts

df_cleaned["abstract_length"] = df_cleaned["abstract"].astype(str).str.len()

# Summary statistics
print("Abstract length statistics:")
print(df_cleaned["abstract_length"].describe())

# Identify very short abstracts
short_abstracts = df_cleaned[
    df_cleaned["abstract_length"] < 200
].sort_values("abstract_length")

print("\nNumber of abstracts shorter than 200 characters:",
      len(short_abstracts))

print("\nShortest abstracts:")
print(
    short_abstracts[
        ["pmid", "title", "abstract_length", "abstract", "layer"]
    ].head(20).to_string(index=False)
)

Abstract length statistics:
count    21256.000000
mean      1729.454930
std        591.863555
min          0.000000
25%       1373.000000
50%       1684.000000
75%       1991.250000
max      14210.000000
Name: abstract_length, dtype: float64

Number of abstracts shorter than 200 characters: 2

Shortest abstracts:
    pmid                                                                                                               title  abstract_length abstract                   layer
41436046                           [The applications and challenges of rapid detection technologies in forensic toxicology].                0          L3_comparative_critical
41372067 [Robots and embodied intelligence in endoscopic transnasal surgery: advances, challenges, and future perspectives].                0          L3_comparative_critical


In [29]:
# Step 13: Investigate the two records with empty abstracts

problem_pmids = ["41436046", "41372067"]

for pmid in problem_pmids:
    print("=" * 100)
    print("PMID:", pmid)

    # Check what was retrieved from PubMed
    retrieved = all_abstracts.get(pmid, None)

    print("\nRetrieved abstract:")
    print(repr(retrieved))

    # Check what exists in the original CSV
    original_record = df[df["pmid"].astype(str) == pmid]

    print("\nOriginal CSV record:")
    if len(original_record) > 0:
        print(original_record[
            ["pmid", "title", "abstract", "year", "journal", "layer"]
        ].to_string(index=False))
    else:
        print("Record not found in original CSV.")

PMID: 41436046

Retrieved abstract:
''

Original CSV record:
    pmid                                                                                     title                                                                                                                        abstract   year          journal                   layer
41436046 [The applications and challenges of rapid detection technologies in forensic toxicology]. 法医毒物学作为法医学的重要分支，是连接中毒事件调查、司法公正裁决与公共安全保障的关键技术支撑，其核心使命在于精准查明中毒案件中的毒物种类、代谢路径、暴露剂量及中毒机制，最终为案件侦破提供方向指引、为司法审判提供科学证据、为突发公共安全事件处置提供决策依据 2025.0 Fa yi xue za zhi L3_comparative_critical
PMID: 41372067

Retrieved abstract:
''

Original CSV record:
    pmid                                                                                                               title                                                                                                                                                                                                            

In [30]:
# Step 14: Restore original abstracts where PubMed returned an empty string

# Identify empty or whitespace-only abstracts
empty_mask = df_cleaned["abstract"].astype(str).str.strip() == ""

print("Empty abstracts before correction:", empty_mask.sum())

# Restore the original CSV abstract for these records
df_cleaned.loc[empty_mask, "abstract"] = df.loc[empty_mask, "abstract"]

# Recalculate abstract lengths
df_cleaned["abstract_length"] = df_cleaned["abstract"].astype(str).str.len()

print("Empty abstracts after correction:",
      (df_cleaned["abstract"].astype(str).str.strip() == "").sum())

print("\nPreviously problematic records after correction:")

print(
    df_cleaned[
        df_cleaned["pmid"].astype(str).isin(["41436046", "41372067"])
    ][["pmid", "title", "abstract_length", "abstract", "layer"]]
    .to_string(index=False)
)

Empty abstracts before correction: 2
Empty abstracts after correction: 0

Previously problematic records after correction:
    pmid                                                                                                               title  abstract_length                                                                                                                                                                                                                                                                                                                                                                               abstract                   layer
41436046                           [The applications and challenges of rapid detection technologies in forensic toxicology].              127                                                                                                                                                                                                   

In [ ]:
# Create a backup of the current abstract column
df["abstract_original"] = df["abstract"]

# Make sure PMIDs are strings
df["pmid"] = df["pmid"].astype(str)

# Replace the existing abstract with the PubMed-retrieved version
# wherever a complete PubMed abstract was successfully retrieved

df["abstract_retrieved"] = df["pmid"].map(all_abstracts)

# Count how many PubMed abstracts were successfully matched
retrieved_count = df["abstract_retrieved"].notna().sum()

print("Total records:", len(df))
print("PubMed abstracts matched:", retrieved_count)
print("PubMed abstracts not matched:", len(df) - retrieved_count)

# Replace only where a PubMed abstract exists and is non-empty
valid_retrieved = (
    df["abstract_retrieved"].notna() &
    (df["abstract_retrieved"].astype(str).str.strip() != "")
)

df.loc[valid_retrieved, "abstract"] = df.loc[valid_retrieved, "abstract_retrieved"]

print("Abstracts replaced with PubMed versions:",
      valid_retrieved.sum())